In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# Load data
data = pd.read_csv('Fertilizer.csv')

# One-hot encoding for Crop Type and Soil Type
crop_type_encoded = pd.get_dummies(data['Crop Type'], prefix='Crop')
soil_type_encoded = pd.get_dummies(data['Soil Type'], prefix='Soil')
data_encoded = pd.concat([data.drop(['Crop Type', 'Soil Type'], axis=1), 
                         crop_type_encoded, soil_type_encoded], axis=1)

# Label encoding for Fertilizer Name
le = LabelEncoder()
data_encoded['Fertilizer Name'] = le.fit_transform(data_encoded['Fertilizer Name'])

# Prepare features (X), regression target (y_usage), and classification target (y_name)
X = data_encoded.drop('Fertilizer Usage (kg/ha)', axis=1)
y_usage = data_encoded['Fertilizer Usage (kg/ha)']
y_name = data_encoded['Fertilizer Name']

# K-means clustering
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
clusters = kmeans.fit_predict(X)
data_encoded['Cluster'] = clusters

# Train models for each cluster and track the best R² for regression
results = {}
best_cluster = None
best_r2 = -float('inf')

for cluster_id in range(optimal_k):
    # Get data for current cluster
    cluster_data = data_encoded[data_encoded['Cluster'] == cluster_id]
    X_cluster = cluster_data.drop(['Fertilizer Usage (kg/ha)', 'Cluster'], axis=1)
    y_usage_cluster = cluster_data['Fertilizer Usage (kg/ha)']
    y_name_cluster = cluster_data['Fertilizer Name']
    
    # Split cluster data
    X_train, X_test, y_usage_train, y_usage_test, y_name_train, y_name_test = train_test_split(
        X_cluster, y_usage_cluster, y_name_cluster, test_size=0.2, random_state=42
    )
    
    # Train XGBoost Regressor for Fertilizer Usage
    xgb_regressor = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
    xgb_regressor.fit(X_train, y_usage_train)
    usage_predictions = xgb_regressor.predict(X_test)
    
    # Train XGBoost Classifier for Fertilizer Name
    xgb_classifier = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
    xgb_classifier.fit(X_train, y_name_train)
    name_predictions = xgb_classifier.predict(X_test)
    
    # Calculate metrics
    mse = mean_squared_error(y_usage_test, usage_predictions)
    r2 = r2_score(y_usage_test, usage_predictions)
    accuracy = accuracy_score(y_name_test, name_predictions)
    
    # Store results
    results[cluster_id] = {
        'mse': mse,
        'r2': r2,
        'accuracy': accuracy,
        'regressor': xgb_regressor,
        'classifier': xgb_classifier,
        'X_test': X_test,
        'y_usage_test': y_usage_test,
        'usage_predictions': usage_predictions,
        'y_name_test': y_name_test,
        'name_predictions': name_predictions
    }
    
    # Track best R²
    if r2 > best_r2:
        best_r2 = r2
        best_mse = mse
        best_cluster = cluster_id
        best_accuracy = accuracy
        best_regressor = xgb_regressor
        best_classifier = xgb_classifier
        best_X_train_columns = X_train.columns  # Store training feature names

# Print best cluster's results
# print(f"Best Cluster: {best_cluster}")

# print(f"Fertilizer Name - Accuracy: {best_accuracy:.2f}")

# Function to preprocess new sample data and predict
def predict_new_sample(sample_data, kmeans, best_regressor, best_classifier, le, training_columns):
    # Convert sample data to DataFrame
    sample_df = pd.DataFrame([sample_data])
    
    # One-hot encode Crop Type and Soil Type
    crop_type_encoded_sample = pd.get_dummies(sample_df['Crop Type'], prefix='Crop')
    soil_type_encoded_sample = pd.get_dummies(sample_df['Soil Type'], prefix='Soil')
    
    # Align with training data columns
    for col in crop_type_encoded.columns:
        if col not in crop_type_encoded_sample.columns:
            crop_type_encoded_sample[col] = 0
    for col in soil_type_encoded.columns:
        if col not in soil_type_encoded_sample.columns:
            soil_type_encoded_sample[col] = 0
    
    # Drop original categorical columns and concatenate encoded ones
    sample_encoded = pd.concat([sample_df.drop(['Crop Type', 'Soil Type'], axis=1), 
                                crop_type_encoded_sample, soil_type_encoded_sample], axis=1)
    
    # Add Fertilizer Name as a placeholder (won’t be used for prediction but must match training structure)
    sample_encoded['Fertilizer Name'] = 0  # Dummy value, will not affect prediction
    
    # Ensure all columns match the training data
    missing_cols = set(training_columns) - set(sample_encoded.columns)
    for col in missing_cols:
        sample_encoded[col] = 0
    sample_encoded = sample_encoded[training_columns]  # Reorder to match training columns
    
    # Predict cluster
    cluster_pred = kmeans.predict(sample_encoded)[0]
    
    # Use best cluster's models if the predicted cluster matches
    if cluster_pred == best_cluster:
        usage_pred = best_regressor.predict(sample_encoded)[0]
        name_pred = le.inverse_transform([best_classifier.predict(sample_encoded)[0]])[0]
    else:
        # Fallback to the specific cluster’s models
        usage_pred = results[cluster_pred]['regressor'].predict(sample_encoded)[0]
        name_pred = le.inverse_transform([results[cluster_pred]['classifier'].predict(sample_encoded)[0]])[0]
    
    return name_pred, usage_pred

# Example new sample data
new_sample = {
    'Temparature': 30.0,
    'Humidity': 60.0,
    'Moisture': 40.0,
    'Soil Type': 'Sandy',
    'Crop Type': 'Maize',
    'Nitrogen': 20,
    'Potassium': 10,
    'Phosphorous': 15
}

# Predict for the new sample
predicted_name, predicted_usage = predict_new_sample(new_sample, kmeans, best_regressor, best_classifier, le, best_X_train_columns)
# print("\nNew Sample Prediction:")
# print(f"Input: {new_sample}")
print(f"Predicted Fertilizer Name: {predicted_name}")
print(f"Predicted Fertilizer Usage (kg/ha): {predicted_usage:.2f}")
print(f"R² Score: {best_r2:.2f}")
print(f"Mean Squared Error: {best_mse:.2f}")